# Label Confusion Sample

The below UUIDs are some process UUIDs from the label set `node_fivedirections_e5_copykatz_0509.csv` corresponding to the PIDSMaker repository / Orthrus paper. 

In [1]:
# UUIDs from node_fivedirections_e5_copykatz_0509.csv from PIDSmaker;
# which should relate to host E5 fivedirections_2 (uuid: 223CBBCE-1BC5-451E-9D2E-5B3A762F31C7)
uuids = {
    "6E9A7E66-BF34-4F21-8E8E-523338C5B529",
    "E9AC31C9-C040-4626-8210-64A39DFAF516",
    "B849F2F5-DA31-4D4F-841A-F88E8D3153B7",
    "28268DF8-EAC3-4C92-A5A0-96F3401CC2FC",
    "819C1F90-B279-4328-8C09-27E794A60994",
    "11E3B2B6-73BB-4A6D-895C-D8690161AC50",
    "495DE842-EC29-45C9-B657-6B5B8554414B",
}

In [2]:
from provenance_explorer.registry.registry_all import get_subdataset_registry
from provenance_explorer.raw_file_handling.dataset_iterator import make_dataset_iterator
from provenance_explorer.raw_file_handling.parsing_helpers import TS_EXTRACTORS, PARSERS
from provenance_explorer.utils.time_conversion import date_string_to_ns_timestamp, ns_timestamp_to_date_string
from datetime import timedelta, timezone

_EDT_OFFSET = timezone(timedelta(hours=-4))

# attack time window of Ground Truth 4.3 attack meant by labelset
start_ns = date_string_to_ns_timestamp("2019-05-09 13:26:00", _EDT_OFFSET)
end_ns = date_string_to_ns_timestamp("2019-05-09 13:56:00", _EDT_OFFSET)

json_parse = PARSERS[("e5","fivedirections")]
custom_parse = lambda c: (c, json_parse(c))
fivedir_raw = make_dataset_iterator(
    get_subdataset_registry("e5","fivedirections"),
    custom_parse,
    TS_EXTRACTORS[("e5","fivedirections")],
    start_ns,
    end_ns
)
host_uuids = set()

for i, (ts, rec) in enumerate(fivedir_raw):
    str_rec, dict_rec = rec
    if len(uuids) == 0:
        break
    try:
        process_uuid = dict_rec["datum"]["com.bbn.tc.schema.avro.cdm20.Subject"]["uuid"]
        if process_uuid in uuids:
            print(str_rec)
            uuids.remove(process_uuid)
            host_uuids.add(dict_rec["hostId"])
    except:
        continue

print()
print(f"Unique Host-IDs of processes labeled malicious: {host_uuids}")


{"datum":{"com.bbn.tc.schema.avro.cdm20.Subject":{"uuid":"E9AC31C9-C040-4626-8210-64A39DFAF516","type":"SUBJECT_PROCESS","cid":6208,"parentSubject":{"com.bbn.tc.schema.avro.cdm20.UUID":"B41EAE5E-AF88-4FA4-B49A-5C179BFE67FD"},"localPrincipal":{"com.bbn.tc.schema.avro.cdm20.UUID":"A48BE6E3-13FF-445E-AF6D-CEF1F530B3EF"},"startTimestampNanos":{"long":1557423721679000000},"unitId":null,"iteration":null,"count":null,"cmdLine":{"string":"\"C:\\Program Files\\mozilla\\firefox\\firefox.exe\" "},"privilegeLevel":null,"importedLibraries":null,"exportedLibraries":null,"properties":null}},"CDMVersion":"20","type":"RECORD_SUBJECT","hostId":"223CBBCE-1BC5-451E-9D2E-5B3A762F31C7","sessionNumber":10513,"source":"SOURCE_WINDOWS_FIVEDIRECTIONS"}
{"datum":{"com.bbn.tc.schema.avro.cdm20.Subject":{"uuid":"6E9A7E66-BF34-4F21-8E8E-523338C5B529","type":"SUBJECT_PROCESS","cid":1460,"parentSubject":{"com.bbn.tc.schema.avro.cdm20.UUID":"09C96C7F-DE86-4496-8087-C2E940244AB6"},"localPrincipal":{"com.bbn.tc.schema.a

Notably, all 3 E5 Fivedirections hosts appear in the labelset, even though the attack only happens on Host 2. Two of the printed samples are Firefox processes. 